# 1. Analisando os dados

## 1.1. Importando Libs

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
from torchmetrics import Accuracy
from torch.utils.data import DataLoader, Dataset


In [ ]:
sys.path.append(str(Path.cwd().parent))

In [ ]:
from src.data_loading import data_download

shakespeare_data = data_download()

In [ ]:
print(shakespeare_data[:175])

## 1.2. Identificando Caracteres e Adicionando ID

In [ ]:
characters = sorted(set(shakespeare_data.lower()))
"".join(characters)

In [ ]:
char_idx = {char : idx for idx, char in enumerate(characters)}
idx_char = {idx : char for idx, char in enumerate(characters)}

In [ ]:
print(f'{idx_char[13]} -> {char_idx['a']}')
print(f'{char_idx['a']} -> {idx_char[13]}')

## 1.3. Criando exemplos de enxode e decode

In [ ]:
from src.text_processing import encode_text

encoded = encode_text('I love NLP!!', char_idx) 
encoded

In [ ]:
from src.text_processing import decode_text

decoded = decode_text(encoded, idx_char)
decoded

_A única perda visível são os caracteres em maiúsculo, visto que esse encoding não é case sensitive_

## 1.4. Criando classe para preparar o dataset

In [ ]:
class CharDataSet(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text, char_idx)
        self.window_length = window_length

    def __len__(self):
        return (len(self.encoded_text) - self.window_length)

    def __getitem__(self, key):
        if key >= len(self):
            raise IndexError('idx out of the range')
        
        end_window = key + self.window_length
        window = self.encoded_text[key : end_window]
        target = self.encoded_text[key + 1 : end_window + 1]

        return window, target

## 1.5. Preparando Dados

- TRAIN: 90%
- VALIDATION: 5%
- TEST: 5%

In [ ]:
window_length = 50

train_set = CharDataSet(shakespeare_data[:1_000_000], window_length)
valid_set = CharDataSet(shakespeare_data[1_000_000:1_060_000], window_length)
test_set = CharDataSet(shakespeare_data[1_060_000:], window_length)

#criando data_loaders
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=128)
test_loader = DataLoader(test_set, batch_size=128)

## 2. Criação e Treinamento do modelo

In [ ]:
from src.model import ShakespeareModel

model = ShakespeareModel(len(characters))

In [ ]:
from src.training import train

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

metric = Accuracy(
    task='multiclass',
    num_classes=len(characters)
)

train(
    model=model,
    epochs=5 ,
    data_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    valid_loader=valid_loader,
    metric=metric,
)

In [ ]:
from src.training import eval

eval(
    model,
    valid_loader,
    metric
)

# 3. Testando o modelo

In [ ]:
model.eval()
text = 'To be or not to b'

#unsqueeze adiciona 1 na dim 0, pois o modelo é batch_first=True
encoded = encode_text(text, char_idx).unsqueeze(dim=0) 
with torch.no_grad():
    logits = model(encoded)

    #[batch, characters, timesteps]
    char_id = logits[0, :, -1].argmax().item()
    pred_char = idx_char[char_id] 

pred_char

## 3.1. Gerando Texto

In [ ]:
from src.generate import append_text

print(append_text(model, text, char_idx, idx_char, num_chars=1000, temperature=0.8))

# 4. Exportando o modelo

In [ ]:
path = Path("../models/shakespeare_char_rnn.pth")

torch.save(model.state_dict(), path)

_para importar esse modelo é só criar um novo modelo ShakespeareModel e utilizar:_

```python
model.load_state_dict(torch.load(path))
```